# Extension 3: Correct SHAP Attribution

This notebook reuses the six trained full-covariate Dunnhumby checkpoints. It does not retrain any model.

The analysis explains first-holdout-week expected frequency and conditional log1p-spend. Each explained household retains its own 80-week transaction history while static and dynamic covariates are integrated over a shared empirical background. These are interventional model attributions conditional on transaction history, not causal campaign effects.

## Failure audit

Earlier attempts failed or produced ambiguous artifacts because:

1. SHAP subprocess failures were hidden or treated as non-fatal.
2. `GradientExplainer` required `(B, 1)` outputs and SHAP versions used different multi-input return layouts.
3. Full-wrapper training mode enabled Transformer dropout while fixing cuDNN LSTM backward.
4. Duplicate runner cells and generic filenames overwrote results across architectures.
5. `N // 2` was labelled a median customer, and one history was used for every household.
6. Dynamic importance used `sum(abs(value))` rather than the additive `abs(sum(value))` grouping.
7. Spend values remained in robust-scaled model units.
8. Only 801 of 2,498 households have observed demographics; missing values were encoded like valid lowest categories.
9. `thesis_final_v2` contains derived results, while checkpoints are stored under `results/final_kaggle/checkpoints/`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_ROOT = PROJECT_ROOT / 'results' / 'final_kaggle'
CHECKPOINT_ROOT = RESULTS_ROOT / 'checkpoints'
TABLES = RESULTS_ROOT / 'tables'
PLOTS = RESULTS_ROOT / 'plots' / 'shap'

checkpoint_names = sorted(path.name for path in CHECKPOINT_ROOT.glob('extension3_*_full_dunnhumby_final_seed*_sample.pt'))
print(f'Found {len(checkpoint_names)} full-covariate checkpoints')
for name in checkpoint_names:
    print(' ', name)
assert len(checkpoint_names) == 6, 'Expected LSTM and Transformer checkpoints for seeds 7, 42, and 2024.'

## Run or resume the analysis

The runner caches completed work using config, checkpoint, sample-manifest, source-code, SHAP-version, and integration-budget hashes. The primary run uses 100 observed-demographic households for the background and a disjoint 100 for explanation. It compares 32 with 64 expected-gradient samples at seed 42 and escalates an architecture to 128 when the convergence or additivity thresholds fail.

In [ ]:
command = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'run_extension3_shap.py'),
    '--results_root', str(RESULTS_ROOT),
    '--checkpoint_root', str(CHECKPOINT_ROOT),
    '--device', 'auto',
]
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## Publication summary

In [ ]:
summary = pd.read_csv(TABLES / 'shap_extension3_summary.csv')
summary.sort_values(['architecture', 'head', 'relative_importance_pct'], ascending=[True, True, False])

In [ ]:
convergence = pd.read_csv(TABLES / 'shap_extension3_convergence.csv')
additivity = pd.read_csv(TABLES / 'shap_extension3_additivity.csv')
display(convergence)
display(additivity)
remaining = additivity[additivity['normalized_additivity_error'] > 0.10]
if len(remaining):
    print('Warning: these runs remain above 10% after the selected convergence budget:')
    display(remaining)

## Primary figures

In [ ]:
for name in [
    'shap_architecture_comparison.png',
    'shap_beeswarm_lstm.png',
    'shap_beeswarm_transformer.png',
    'shap_temporal_lstm.png',
    'shap_temporal_transformer.png',
]:
    print(name)
    display(Image(filename=PLOTS / name))

## Missing-demographic sensitivity

This seed-42 sensitivity includes all households. It must not replace the primary observed-demographic analysis because absent demographic rows are encoded as zero and are confounded with valid lowest income and household-size categories.

In [ ]:
sensitivity = pd.read_csv(TABLES / 'shap_extension3_sensitivity_all_households.csv')
sensitivity.sort_values(['architecture', 'head', 'relative_importance_pct'], ascending=[True, True, False])

In [ ]:
manifest = json.loads((TABLES / 'shap_extension3_run_manifest.json').read_text())
manifest